# Round 3 — THE FINAL MODEL

**This is the run Round 3 is graded on.** Not an arm. The arms are finished ([`docs/rung3/levers.md`](../docs/rung3/levers.md)): the scan profile was a null, one-measure-per-strip was dropped, and the staccato distractor **passed**.

## What this run consumes — settled 2026-08-31, closed to additions

| role | pool |
|---|---|
| synthetic training | **`strips_v7_final`** — `pieces_v4.json` rendered with **three** flags: `--staccato-noise --concave-tuplet --usul-barline` |
| real training | **`strips_b8`** and nothing else — 3,936 strips, 3,546 train-side, on the CURRENT crops |
| split | `data/split_v4.json` |
| selection | `_realval_v2` (+ `_tupletval`), read on the Mac |
| grading | `examv3`, read ONCE, on the Mac |

⛔ **`strips_nota` / `strips_r1` / `strips_tup` MUST NOT APPEAR IN THIS NOTEBOOK.** They are the same music on the **retired** slicer's crops, superseded by `strips_b8`. Passing them trains the graded model on crops the app no longer cuts, and **nothing downstream would notice** — the filenames survive a re-slice even though the pixels do not.

⛔ **`b8-review`, the old human fixes, `batch3` and `reslice-all` are ROUND 4.** The last two have no promotion path at all: 53 of `batch3`'s 66 corrections and all 50 of `reslice-all`'s sit on strips the emitter dropped.

## Why `:5` and not `:9`

The `:N` suffix oversamples a real pool so real is about a third of stage-2 batches. That share is the recipe; `N` is only how it is reached, and it has to move when the pool size moves.

| pool | repeat | real share of stage-2 batches |
|---|---|---|
| old pools (2,059 train) | `:9` | 33.9% — the figure on record |
| **`strips_b8` (3,546 train)** | **`:5`** | **33.0%** ✅ |
| `strips_b8` | `:9` | 47.0% ⛔ — changes the recipe as a side effect |

Measured 2026-08-31 against 36,057 train-side synthetic strips. ⚠ **`train.py` prints the real pool counts at startup — read that line rather than trusting this table.**

## The recipe is held fixed from the arms

Stage 1: **6,000 steps @ batch 16**, synthetic only, from BASE. Stage 2: **2,000 steps @ batch 16** from stage 1, `--every-share 0.15`. ⚠ **Do not pass `--photo-share` or `--scan-share`** — the mix is `train.py`'s default (screenshot 0.65 / photo 0.35), and the scan profile was settled **off** on 2026-08-19 after arm 1 came back null.

## Three flags in one render

⚠ A general movement in this run is **not attributable to any one flag**. That was accepted when the render was specified. Two of the three keep their own paired instrument on the Mac — the staccato arm's false-dot scorer, and the dotted barline's false-`\repstart` scorer — so those two survive it. The concave tuplet mark does not and never claimed to.

## Both checkpoints come home

`best` is selected on a val loss that is **~94.6% synthetic**, in a round graded on real pages. Rather than swap the selector mid-round, this run keeps it **and** brings `last` home, and the two are compared on `_realval_v2` **before** the exam. Legal by the standing rule: real-val selects, the exam is one-shot. ([`docs/BACKLOG.md`](../docs/BACKLOG.md) item 3.)

⚠ **Exam strips are not on this VM and the exam is not read here.** One shot, later, on the Mac.


In [ ]:
# ===== THE ONLY KNOBS IN THIS NOTEBOOK — and the mix is NOT one of them =====
ARM = 'final'
STRIPS = 'data/synthetic/strips_v7_final'   # the 3-flag render: staccato + concave tuplet + usul barline
ZIP = 'tnc_round3_final_colab.zip'
DRIVE = '/content/drive/MyDrive/tnc'
# THE REAL POOL. One pool, current crops. ⛔ Never add strips_nota / strips_r1 / strips_tup.
REAL = 'data/real/rung3/strips_b8'
REPEAT = 5          # -> real ~33% of stage-2 batches at 3,546 train-side strips. See the header.
print(ARM, STRIPS, ZIP, f'| real: {REAL}:{REPEAT} | mix: train.py defaults (0.65/0.35, scan off)')

In [ ]:
# Which GPU did we get? (T4 16GB / L4 24GB / A100 40GB)
!nvidia-smi

In [ ]:
# Mount Google Drive (approve the popup).
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%time
# Copy the package Drive -> VM disk and unzip (fast local disk for the dataloader).
!cp {DRIVE}/{ZIP} /content/
!rm -rf /content/tnc && mkdir /content/tnc
!cd /content/tnc && unzip -q /content/{ZIP}

# WHICH CORPUS IS ACTUALLY ON DISK. All three flags are LABEL-FREE — they change pixels only, so the
# manifest cannot tell this corpus from an arm's. render_config.json is the ONLY place they are
# checkable, which is why they are asserted here as well as in make_round3_colab_zip.sh.
import json
cfg = json.load(open(f'/content/tnc/{STRIPS}/render_config.json'))
print(cfg)
assert cfg.get('staccatoNoise') is True,  'MISSING --staccato-noise — this is not the final render'
assert cfg.get('concaveTuplet') is True,  'MISSING --concave-tuplet — this is not the final render'
assert cfg.get('usulBarline') is True,    'MISSING --usul-barline — this is not the final render'
assert cfg['legacyTupletMark'] is False and cfg['thinSharps'] is True and cfg['printNoise'] is False
assert cfg.get('maxMeasures', None) is None

# WHICH REAL POOL IS ON DISK. The retired pools' filenames survive a re-slice and their pixels do
# not, so a name check is not enough on its own — but their ABSENCE is decisive.
import os
assert os.path.isdir(f'/content/tnc/{REAL}'), f'{REAL} missing from the zip'
for dead in ('strips_nota', 'strips_r1', 'strips_tup'):
    assert not os.path.isdir(f'/content/tnc/data/real/rung3/{dead}'), \
        f'RETIRED POOL {dead} IS IN THE ZIP — rebuild it with: sh scripts/make_round3_colab_zip.sh final'
!wc -l /content/tnc/{STRIPS}/manifest.jsonl
!wc -l /content/tnc/{REAL}/manifest.jsonl   # expect 3936
!python -c "import json;s=json.load(open('/content/tnc/data/split_v4.json'));print('train',len(s['train_pieces']),'val',len(s['val_pieces']))"

In [ ]:
# Dependencies (torch + torchvision are preinstalled on Colab).
!pip -q install transformers albumentations opencv-python-headless

In [ ]:
# ===== THE FLAGS ARE THE RENDER — prove they are in the PIXELS before spending a GPU hour =====
# Nothing downstream records them: labels, manifest and split are what they would be with the flags
# off (188 strip labels over 4 scores are byte-identical with --usul-barline on and off). So the
# check is on the IMAGES, and on the mix being the default.
%cd /content/tnc
import sys
sys.path.insert(0, 'src/vision')
from augment import Augmenter
a = Augmenter(seed=7)
assert (a.photo_share, a.scan_share) == (0.35, 0.0), (a.photo_share, a.scan_share)
print(f'mix: screenshot {1-a.photo_share-a.scan_share:.2f} / photo {a.photo_share} / scan {a.scan_share}  (default)')

import json, random
from PIL import Image
rows = [json.loads(l) for l in open(f'{STRIPS}/manifest.jsonl')]
print(f'{len(rows)} synthetic strips')

# LOOK AT THEM. Each flag is coined PER PIECE, so a sample of one piece shows nothing — these draw
# from strips whose label gives the flag something to act on.
def show(pred, what, n=2):
    hits = [r for r in rows if pred(r)]
    print(f'{what}: {len(hits)} candidate strips')
    random.Random(7).shuffle(hits)
    for r in hits[:n]:
        print('  ', r['image'])
        display(Image.open(f"{STRIPS}/{r['image']}"))

show(lambda r: '.' in r['label'], 'staccato — dots ABOVE/BELOW noteheads, not beside')
show(lambda r: '\\tup3' in r['label'], 'tuplet — some pieces draw a CONTINUOUS arc with the 3 inside it')
show(lambda r: '|' in r['label'], 'usul barline — light DASHED rules INSIDE the bar, at the beat groups')

In [ ]:
# SHAKEOUT (~3 min): 150 tiny steps from BASE — a WIRING smoke, not a result.
# Expect: `vocab: +25 tokens -> 100 ids`, ONE real pool listed, `exam-disjointness OK`,
# `augment=on (screenshot 0.65 / photo 0.35)`, and val loss FALLING.
# ⚠ READ THE `real pool ... : N train xR / M val strips` LINE — that is the only place the pool and
# its repeat are visible, and it is what the header's 33% rests on.
%cd /content/tnc
!python -u src/vision/train.py --strips-dir {STRIPS} --split data/split_v4.json \
    --real-dir {REAL} \
    --every-share 0.15 --out-dir /content/r3-shakeout \
    --lr 3e-5 --warmup-steps 30 --max-steps 150 --batch-size 8 \
    --limit-val 40 --eval-every 50 --save-every 50 --log-every 25 --num-workers 2

In [ ]:
# ===== CALIBRATE THROUGHPUT ON *THIS* RUNTIME (~2-3 min) — before any long run =====
#   hours = (steps * batch) / samples_per_sec / 3600
# ⚠ Whatever you set, the STEP COUNTS AND BATCH SIZE must match the arms': 6000 @ 16, then 2000 @ 16.
# --num-workers and the GPU model do not change the result; those two do.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!nproc
%cd /content/tnc
!python -u src/vision/train.py --strips-dir {STRIPS} --split data/split_v4.json \
    --real-dir {REAL} \
    --every-share 0.15 --out-dir /content/calib \
    --lr 3e-5 --warmup-steps 20 --max-steps 60 --batch-size 16 \
    --limit-val 8 --eval-every 60 --save-every 60 --log-every 20 --num-workers 10

In [ ]:
# ===== STAGE 1 — carry-dominant SYNTHETIC ONLY, from BASE =====
# No --real-dir: this builds the carry-native synthetic checkpoint stage 2 specialises. It is also
# where the three flags do their work, because real strips train CLEAN (--augment-real is off, and
# stays off — double-degrading a blurry nota scan buries its signal).
# ⚠ `-u` is not cosmetic: Colab block-buffers a subprocess's stdout, so without it the log lines sit
# in an 8 KB buffer and a healthy run looks frozen for minutes at a time.
%cd /content/tnc
!python -u src/vision/train.py --strips-dir {STRIPS} --split data/split_v4.json \
    --every-share 0.15 --out-dir {DRIVE}/r3-{ARM}-stage1 \
    --lr 3e-5 --max-steps 6000 --batch-size 16 --num-workers 10  # ~nproc-2; T4 (2 vCPU) use 2

In [ ]:
# ===== STAGE 2 — real-SPECIALISATION fine-tune from stage 1 =====
# Fresh LOW lr + short warmup from the stage-1 checkpoint.
# ⚠ `:5`, NOT `:9`. The recipe is "real ~1/3 of batches"; the repeat is only how that is reached, and
# strips_b8 is 72% larger than the pools `:9` was chosen for. `:9` here would put real at 47%.
%cd /content/tnc
!python -u src/vision/train.py --model {DRIVE}/r3-{ARM}-stage1/best \
    --strips-dir {STRIPS} --split data/split_v4.json \
    --real-dir {REAL}:{REPEAT} \
    --every-share 0.15 --out-dir {DRIVE}/r3-{ARM}-stage2 \
    --lr 1e-5 --warmup-steps 100 --max-steps 2000 --batch-size 16 --num-workers 10

In [ ]:
# RESUME after a disconnect: re-run the setup cells, then this with the SAME flags as the stage
# you were running (edit out-dir/flags to match). --resume reloads model+optimizer+scheduler from
# <out-dir>/last and ignores --model.
%cd /content/tnc
!python -u src/vision/train.py --strips-dir {STRIPS} --split data/split_v4.json \
    --every-share 0.15 --out-dir {DRIVE}/r3-{ARM}-stage1 \
    --lr 3e-5 --max-steps 6000 --batch-size 16 --num-workers 10 --resume

In [ ]:
# ===== SANITY ONLY — did anything break? =====
# NOT the pre-registered number, and NOT the selection. Both are read on the Mac. This cell exists so
# a broken run is caught before it is downloaded, and to see `best` and `last` side by side.
%cd /content/tnc
!python src/vision/make_realval_pool.py --real-dir {REAL} --split data/split_v4.json

for ck in [f'r3-{ARM}-stage2/best', f'r3-{ARM}-stage2/last']:
    print('=' * 70, '\n==', ck)
    !python src/vision/eval_omr.py --checkpoint {DRIVE}/{ck} \
        --strips-dir data/real/rung3/_realval --split none --show-errors 0

## After the run — the order matters, and the exam is LAST

1. **Download BOTH stage-2 checkpoints** from `MyDrive/tnc/r3-final-stage2/` into `data/checkpoints/` on the Mac — `best` **and** `last`. Stage 2 selects `best` on a synth-dominated val mix, which is the selector Lever 5 already distrusts, and arm 1 showed the two can disagree.

2. **Choose between them on `_realval_v2`** — real-val **selects**, which is legal; it does not grade.
   ```bash
   .venv-ml/bin/python scripts/rung3/paired_arm_score.py \
       --ctl data/checkpoints/round2-stage2-best --arm data/checkpoints/r3-final-stage2-best \
       --pool data/real/rung3/_realval_v2 --out data/real/rung3/final/best.json
   ```
   ⚠ Real-val **orders** models; it does not predict the exam. Round 1 it was out by **28 points**.

3. **Read the two attributable flags on their own paired instruments**, before anything general:
   ```bash
   .venv-ml/bin/python scripts/rung3/staccato_falsedot_score.py \
       --checkpoint data/checkpoints/r3-final-stage2-best --compare data/checkpoints/round2-stage2-best
   .venv-ml/bin/python scripts/rung3/usul_falserep_score.py \
       --checkpoint data/checkpoints/r3-final-stage2-best --compare data/checkpoints/round2-stage2-best
   ```

4. ⛔ **Settle what 75% means BEFORE the exam is read** — absolute, or re-expressed against `round2-stage2-best` re-measured on the rebuilt exam. **Choosing after seeing the number is the one option that is not available.** [`docs/rung3/round3-criteria.md`](../docs/rung3/round3-criteria.md) §3c.

5. ⛔ **`round2-stage2-best` must already be re-scored on `examv3`** — it is the baseline column of every floor pair and a **precondition of the read**. The rebuilt exam is harder than the one the floor was signed against (~12 candidate strips a page against 7.1), so the old number does not carry over.

6. **Then the exam, ONCE.** Report the primary **with its interval** — at ~63 pages the 95% half-width is about ±10-12 points. Add the free column: split `\tup3` recall by first-in-strip vs later.

7. **A miss is a miss.** The bar was signed before training and is not re-opened after the read; the public launch waits for Round 4.
